# Here, we will demonstrate Reflexion

We use the Solver that we created earlier and instead of using the Rejector, we use the Reflector and show the steps of Reflexion.

In [ ]:
from multi_agent.multi_agent import Problem
import importlib
import reflexion
import prompt_template
importlib.reload(reflexion)
importlib.reload(prompt_template)
from prompt_template import Reflector, Solver, Evaluator

q = """
Define $\\operatorname{sgn}(x)$ to be $1$ when $x$ is positive, $-1$ when $x$ is $0$.
Compute $$ \\sum_{n=1}^{\\infty} \\frac{\\operatorname{sgn}\\left(\\sin\\left(2^{n}\\right)\\right)}{2^{n}} $$
(The arguments to sin are in radians.)
"""

a = "answer 1-\\frac{2}{\\pi}"

problem = problem = Problem(
    roles=[Solver, Reflector],  # add Solver if you have one
    problem_descr=q,
    answer=a
)

print(problem)

Welcome Solver, and Reflector. Together, you should solve the following problem:
>>  Define $\operatorname{sgn}(x)$ to be $1$ when $x$ is positive, $-1$ when $x$
is $0$. Compute $$ \sum_{n=1}^{\infty}
\frac{\operatorname{sgn}\left(\sin\left(2^{n}\right)\right)}{2^{n}} $$ (The
arguments to sin are in radians.) .<<  "When you are done, you should submidt
your answer as: ANSWER: <your answer>.  No latex formatting, just the raw
number/numbers or strings at the very end.  Before you start sharing your
toughts, give a little summary of the conversation so far.  Give a list of the
currently suggested answers. Everytime you propose an aswer, check this list.
You proposal cannot be in this this list. Try again and submit a new unique
answer."


In [ ]:
from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)


In [ ]:
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)


=== Attempt 1 ===
We interpret sgn as the usual sign function: sgn(x) = 1 if x > 0, -1 if x < 0 (and 0 if x = 0). Since sin(2^n) ≠ 0 for every integer n (because 2^n is an integer and π is irrational, so 2^n ≠ kπ), we have sgn(sin(2^n)) ∈ {+1, -1} for all n.

1) Sign criterion
Sin is positive on intervals (2kπ, (2k+1)π) and negative on ((2k+1)π, (2k+2)π). Therefore
sgn(sin x) = +1 if x mod 2π ∈ (0, π), and -1 if x mod 2π ∈ (π, 2π).

Thus for our sequence, define t_n ≡ 2^n mod 2π. Then
ε_n := sgn(sin(2^n)) = +1 if t_n ∈ (0, π), and ε_n = -1 if t_n ∈ (π, 2π).

Also t_{n+1} ≡ 2 t_n (mod 2π). Start with t_1 ≡ 2 (mod 2π).

2) Numeric computation
Using the doubling recurrence t_{n+1} ≡ 2 t_n mod 2π, the signs ε_n up to, say, n = 30 are obtained as follows (positive means sin > 0):

n :  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30
ε_n: +  -  +  -  +  +  +  -  +  -  -  -  -  -  +  +  -  -  +  +  +  -  -  +  +  +  -  +  -

3) Partial sums and the l

In [4]:
from reflexion import ReflexionAgent, ReflexionStrategy
from functools import partial
eval = partial(evaluator_fn, model=model)

# from https://www.testprepreview.com/modules/mathematics1.htm
problem_descr = """ 
An instrument store gives a 10 percent discount to all students off the original cost of an instrument. During a back to school sale an additional 15 percent is taken off the discounted price. Julie, a student at the local high school, purchases a flute for $306. How much did it originally cost?
"""

ans = 400

problem1 = problem = Problem(
    roles=[Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_descr,
    answer=ans
)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem1.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
- Let the original price be P dollars.
- After the 10% student discount: price = 0.9P.
- After the additional 15% discount on the discounted price: price = 0.85 × 0.9P = 0.765P.
- This final price equals $306, so 0.765P = 306.
- Solve for P: P = 306 / 0.765 = 400.

Original cost: $400.
Score: true
Feedback: - Verdict: True

- Feedback:
  - The method is correct: the final price factor is 0.9 × 0.85 = 0.765, so 0.765P = 306, giving P = 306 / 0.765 = 400.
  - Quick check confirms the result: 400 original → 10% off → 360; 15% off → 306.
  - For clarity, you could present the discount product as an explicit equation: 306 = P × (9/10) × (17/20), then P = 306 × 200 / 153 = 400.
  - A concise write-up might start with the combined discount factor and then show the division to find the original price, followed by a brief verification.
true

FINAL SOLUTION:
 - Let the original price be P dollars.
- After the 10% student discount: price = 0.9P.
- After the additional 15% disco